# 实战案例 5：RL 项目落地 Checklist 走查（以 CartPole 为例）

本 Notebook 将 [RL-07-05-实战-项目Checklist](../RL-07-05-实战-项目Checklist.md) 的立项五问、评估门禁落实到**可执行代码与检查项**。

案例：内部「平衡控制 POC」——在仿真中验证 PPO，再决定是否采购硬件。

## Checklist 1：MDP 是否闭环？

In [ ]:
MDP_SPEC = {
    "S": "obs[0:4] = cart_pos, cart_vel, pole_angle, pole_vel",
    "A": "Discrete(2): 0=左推, 1=右推",
    "R": "每步 +1 直至失败",
    "gamma": 0.99,
    "episode_end": "杆倾角/位移超界",
}
for k, v in MDP_SPEC.items():
    print(f"{k}: {v}")
print("\n[✓] MDP 闭环：Gymnasium CartPole-v1 已形式化")

## Checklist 2：奖励与 KPI 对齐？

In [ ]:
KPI = {"业务": "杆不倒时间最大化", "RL奖励": "每步+1", "对齐": True}
ANTI_PATTERN = "若奖励改为仅终点+100，中间步0 → 信用分配困难"
print(KPI); print("反例:", ANTI_PATTERN)

## Checklist 3：Baseline + 多 seed 评估

In [ ]:
!pip install -q gymnasium stable-baselines3 numpy

In [ ]:
import gymnasium as gym
import numpy as np
from stable_baselines3 import PPO
from stable_baselines3.common.evaluation import evaluate_policy

SEEDS = [0, 1, 2]
STEPS = 30_000
results = []
for seed in SEEDS:
    env = gym.make("CartPole-v1")
    model = PPO("MlpPolicy", env, seed=seed, verbose=0)
    model.learn(total_timesteps=STEPS)
    mean, std = evaluate_policy(model, env, n_eval_episodes=10, deterministic=True)
    results.append(mean)
    env.close()
    print(f"seed={seed}: eval return = {mean:.1f} ± {std:.1f}")
print(f"\n汇总: {np.mean(results):.1f} ± {np.std(results):.1f}")
GATE = np.mean(results) > 350
print(f"上线门禁（POC）通过: {GATE}")

## Checklist 4：实验记录

生产环境应写入 `config.yaml` + git commit + requirements.txt。

In [ ]:
import json, datetime
record = {
    "project": "cartpole_balance_poc",
    "date": datetime.date.today().isoformat(),
    "algo": "PPO",
    "total_steps": STEPS,
    "seeds": SEEDS,
    "eval_returns": results,
    "gate_passed": bool(GATE),
    "next_step": "域随机化 Wrapper → 硬件在环" if GATE else "增加步数/调参",
}
print(json.dumps(record, indent=2, ensure_ascii=False))

## 交付物清单

- [x] MDP 文档
- [x] Baseline 曲线
- [x] 多 seed eval
- [ ] 影子模式（真机阶段）
- [ ] 回滚方案

完整案例代码见同目录 [cartpole_control_baseline.ipynb](../RL-07-01-实战-CartPole到MuJoCo/cartpole_control_baseline.ipynb)。